# RideOps Support — Exploratory Data Analysis & Root-Cause Investigation

**Goal:** Understand why the support-ticket backlog is growing, identify the strongest evidence for the underlying bottleneck, and use that evidence to choose a focused v0 to build.

> This notebook is deliberately an **investigation notebook**. Do not decide the product solution before the data supports a problem worth solving.

### Investigation flow
1. Understand the dataset and ticket lifecycle
2. Check data quality and timestamps
3. Establish ticket-volume and open-backlog trends
4. Analyze resolution/closure performance
5. Investigate escalation to Engineering
6. Investigate reopens and repeated handling
7. Compare categories, priorities, clients, and agents
8. Examine CSAT
9. Build an evidence table of candidate root causes
10. Choose the problem, user, and success metric for the v0


## 1. Imports and notebook settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

plt.rcParams["figure.figsize"] = (10, 5)

TICKETS_PATH = "../data/rideops_support_tickets.csv"
DICT_PATH = "../data/rideops_data_dictionary.csv"

tickets = pd.read_csv(TICKETS_PATH)
data_dict = pd.read_csv(DICT_PATH)

print(f"Tickets shape: {tickets.shape}")
print(f"Dictionary shape: {data_dict.shape}")


: 

## 2. Understand the data dictionary

Start with the dictionary before doing analysis. It tells us what each field means and prevents us from inventing interpretations.

In [ ]:
display(data_dict)

print("\nTicket columns:")
for i, c in enumerate(tickets.columns, 1):
    print(f"{i:2}. {c}")


## 3. First look at the ticket data

In [ ]:
display(tickets.head())
display(tickets.sample(min(5, len(tickets)), random_state=42))

print("\nData types:")
display(tickets.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = tickets.isna().sum().to_frame("missing")
missing["missing_pct"] = missing["missing"] / len(tickets) * 100
display(missing.sort_values("missing_pct", ascending=False))

print("\nDuplicate rows:", tickets.duplicated().sum())


## 4. Parse timestamps and inspect the ticket lifecycle

The exact timestamp columns are discovered from the column names. This section converts timestamp-like fields to datetime where possible.

The lifecycle we care about is approximately:

**Created → L1 handling → Escalation (if any) → Engineering → Resolution/Return → Closure → Possible Reopen**

The analysis should use the actual fields in the dictionary rather than assuming every ticket follows every stage.

In [ ]:
# Convert columns that look like timestamps/dates.
timestamp_candidates = []
for col in tickets.columns:
    name = col.lower()
    if any(k in name for k in ["timestamp", "date", "time", "created", "closed", "resolved", "escalat", "reopen"]):
        timestamp_candidates.append(col)

for col in timestamp_candidates:
    converted = pd.to_datetime(tickets[col], errors="coerce")
    # Only replace when at least some values successfully parse.
    if converted.notna().sum() > 0:
        tickets[col] = converted

print("Timestamp/date-like columns:")
for col in timestamp_candidates:
    print(f"- {col}: {tickets[col].dtype}")

display(tickets.describe(include="all").T)


## 5. Establish the basic ticket picture

These are the baseline numbers we will repeatedly refer back to.

Important distinction:

- **Ticket volume** = how many tickets arrive
- **Closed volume** = how many leave the system
- **Open backlog** = tickets still open at the data snapshot

Do not conclude that one of these is the root cause yet.

In [ ]:
print("Total tickets:", len(tickets))

for col in ["status", "priority", "category", "escalated", "escalation_status", "engineering_outcome"]:
    if col in tickets.columns:
        print(f"\n--- {col} ---")
        display(tickets[col].value_counts(dropna=False).to_frame("count"))

if "status" in tickets.columns:
    print("\nStatus percentages:")
    display((tickets["status"].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame("pct"))


## 6. Ticket arrivals over time

First ask: **is the incoming workload itself rising?**

Use weekly counts so that a single unusual day does not dominate the picture.

In [ ]:
date_col = next((c for c in tickets.columns if c.lower() in ["created_at", "created_time", "ticket_created_at", "created_timestamp"]), None)

if date_col is None:
    datetime_cols = [c for c in tickets.columns if pd.api.types.is_datetime64_any_dtype(tickets[c])]
    date_col = datetime_cols[0] if datetime_cols else None

print("Using creation-time column:", date_col)

if date_col:
    weekly_arrivals = (
        tickets.dropna(subset=[date_col])
        .set_index(date_col)
        .resample("W-MON")
        .size()
        .rename("tickets")
    )
    display(weekly_arrivals.to_frame())

    weekly_arrivals.plot(kind="line", marker="o", title="Weekly Ticket Arrivals")
    plt.xlabel("Week")
    plt.ylabel("Tickets")
    plt.tight_layout()
    plt.show()
else:
    print("No creation timestamp could be identified automatically.")


## 7. Open backlog and aging

The dataset is a snapshot pulled on 30 September 2026. For tickets still open, age them from creation to the snapshot date.

This answers a key question:

> **Are tickets merely numerous, or are a meaningful number of them becoming old?**

In [ ]:
SNAPSHOT_DATE = pd.Timestamp("2026-09-30 23:59:59")

if date_col:
    tickets["ticket_age_hours_at_snapshot"] = (
        SNAPSHOT_DATE - tickets[date_col]
    ).dt.total_seconds() / 3600

if "status" in tickets.columns:
    open_mask = tickets["status"].astype(str).str.lower().str.contains("open")
    open_tickets = tickets.loc[open_mask].copy()
    print("Open tickets:", len(open_tickets))

    if "ticket_age_hours_at_snapshot" in open_tickets:
        display(open_tickets["ticket_age_hours_at_snapshot"].describe(percentiles=[.5, .75, .9, .95]))

        open_tickets["age_bucket"] = pd.cut(
            open_tickets["ticket_age_hours_at_snapshot"],
            bins=[-np.inf, 4, 8, 24, 48, 72, 168, np.inf],
            labels=["<4h", "4-8h", "8-24h", "1-2d", "2-3d", "3-7d", ">7d"]
        )
        display(open_tickets["age_bucket"].value_counts().sort_index().to_frame("open_tickets"))

        open_tickets["age_bucket"].value_counts().sort_index().plot(kind="bar", title="Open Tickets by Age")
        plt.xlabel("Age")
        plt.ylabel("Open tickets")
        plt.tight_layout()
        plt.show()


## 8. Resolution / closure time

If the required timestamps exist, calculate time from creation to resolution/closure.

Use **median and percentiles**, not only the average, because support times can have long tails.

Questions:
- What is the typical resolution time?
- How large is the long tail?
- Does resolution time differ by priority/category?


In [ ]:
def find_col(candidates):
    lower_map = {c.lower(): c for c in tickets.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

resolved_col = find_col(["resolved_at", "resolution_timestamp", "resolved_timestamp", "resolution_time"])
closed_col = find_col(["closed_at", "closure_timestamp", "closed_timestamp"])

end_col = resolved_col or closed_col

if date_col and end_col:
    tickets["resolution_hours"] = (
        tickets[end_col] - tickets[date_col]
    ).dt.total_seconds() / 3600

    valid_resolution = tickets["resolution_hours"].dropna()
    print("Using:", end_col)
    display(valid_resolution.describe(percentiles=[.5, .75, .9, .95, .99]).to_frame("hours"))
else:
    print("Could not identify both creation and resolution/closure timestamps automatically.")


## 9. Escalation analysis — the Meera vs Sandeep question

This is a central hypothesis in the case.

We want to establish:

1. How often does L1 escalate?
2. What happens after escalation?
3. How long do escalated tickets take?
4. How often are escalated tickets returned to L1?
5. Are escalations concentrated in particular categories/priorities/clients?

**Do not automatically conclude that escalation is bad.** A high escalation rate can be appropriate if the underlying issues genuinely require Engineering.

In [ ]:
# Flexible discovery of likely escalation-related columns.
esc_cols = [c for c in tickets.columns if any(k in c.lower() for k in ["escalat", "engineer", "return"])]
print("Potential escalation/engineering columns:")
for c in esc_cols:
    print("-", c)

for c in esc_cols:
    print(f"\n--- {c} ---")
    display(tickets[c].value_counts(dropna=False).to_frame("count").head(20))


In [ ]:
# Basic escalation-rate calculation using the actual escalated flag if available.
esc_flag = find_col(["escalated", "is_escalated", "escalation_flag"])

if esc_flag:
    esc = tickets[esc_flag].astype(str).str.strip().str.lower().isin(
        ["yes", "true", "1", "y", "escalated"]
    )
    print("Escalated tickets:", esc.sum())
    print("Escalation rate:", round(esc.mean() * 100, 2), "%")
else:
    print("No standard escalated flag found automatically.")

# Look for explicit return-to-L1/outcome fields.
for c in tickets.columns:
    if any(k in c.lower() for k in ["return", "engineering_outcome", "eng_outcome"]):
        print(f"\n{c} value counts:")
        display(tickets[c].value_counts(dropna=False).to_frame("count"))


## 10. Escalation time and engineering aging

If escalation and engineering timestamps are available, measure the time spent in the engineering stage.

This helps distinguish:

**"Too many tickets are being escalated"** from **"Engineering is taking too long."**

In [ ]:
# Print all timestamp columns to make the engineering-stage calculation transparent.
dt_cols = [c for c in tickets.columns if pd.api.types.is_datetime64_any_dtype(tickets[c])]
print("Datetime columns:")
for c in dt_cols:
    print("-", c)

# Candidate pairs are detected by names. If your dictionary uses different names,
# this is the one section to adjust after inspecting the printed columns.
def first_matching(patterns):
    for c in tickets.columns:
        lc = c.lower()
        if all(p in lc for p in patterns):
            return c
    return None

esc_time = first_matching(["escalat"])
eng_end = first_matching(["engineer", "resolved"]) or first_matching(["engineer", "closed"])

print("\nCandidate escalation timestamp:", esc_time)
print("Candidate engineering end timestamp:", eng_end)

if esc_time and eng_end and pd.api.types.is_datetime64_any_dtype(tickets[esc_time]) and pd.api.types.is_datetime64_any_dtype(tickets[eng_end]):
    tickets["engineering_stage_hours"] = (
        tickets[eng_end] - tickets[esc_time]
    ).dt.total_seconds() / 3600
    display(tickets["engineering_stage_hours"].describe(percentiles=[.5,.75,.9,.95,.99]).to_frame("hours"))
else:
    print("Automatic engineering-duration calculation was not possible; inspect the datetime columns above.")


## 11. Reopens

A ticket that is reopened represents additional work after an apparent resolution.

Ask:
- How common are reopens?
- Which categories have the highest reopen rate?
- Are escalated tickets more likely to reopen?
- Is there a relationship between reopens and CSAT?


In [ ]:
reopen_col = find_col(["reopened", "reopen_count", "reopens", "number_of_reopens"])

if reopen_col:
    print("Using reopen field:", reopen_col)
    display(tickets[reopen_col].value_counts(dropna=False).to_frame("count"))

    if pd.api.types.is_numeric_dtype(tickets[reopen_col]):
        tickets["has_reopen"] = tickets[reopen_col].fillna(0) > 0
    else:
        tickets["has_reopen"] = tickets[reopen_col].astype(str).str.lower().isin(
            ["yes", "true", "1", "y"]
        )

    print("Reopen rate:", round(tickets["has_reopen"].mean() * 100, 2), "%")
else:
    print("No standard reopen field found automatically.")


## 12. Category × priority × escalation

Averages across the entire support system can hide the real problem.

Find combinations that are both:
- frequent, and
- operationally expensive (slow, escalated, reopened, or low-CSAT).



In [ ]:
def show_group_metrics(group_cols, min_count=5):
    cols = [c for c in group_cols if c in tickets.columns]
    if not cols:
        print("No matching columns found.")
        return
    agg = tickets.groupby(cols, dropna=False).agg(
        tickets=(tickets.columns[0], "size")
    )

    if "resolution_hours" in tickets.columns:
        agg["median_resolution_h"] = tickets.groupby(cols, dropna=False)["resolution_hours"].median()
    if esc_flag:
        agg["escalation_rate_pct"] = tickets.groupby(cols, dropna=False)[esc_flag].apply(
            lambda s: s.astype(str).str.lower().isin(["yes","true","1","y","escalated"]).mean()*100
        )
    if "has_reopen" in tickets.columns:
        agg["reopen_rate_pct"] = tickets.groupby(cols, dropna=False)["has_reopen"].mean()*100

    agg = agg[agg["tickets"] >= min_count].sort_values("tickets", ascending=False)
    display(agg)

show_group_metrics(["category"])
show_group_metrics(["category", "priority"])


## 13. Client concentration

The objective is not to blame clients. We want to know whether a small number of clients or client-specific patterns explain a large share of the workload.

Look for:
- ticket concentration
- escalation concentration
- resolution-time differences
- recurring categories


In [ ]:
client_col = find_col(["client", "client_name", "customer", "customer_name"])

if client_col:
    client_summary = tickets.groupby(client_col, dropna=False).agg(
        tickets=(tickets.columns[0], "size")
    )

    if "resolution_hours" in tickets.columns:
        client_summary["median_resolution_h"] = tickets.groupby(client_col)["resolution_hours"].median()

    if esc_flag:
        client_summary["escalation_rate_pct"] = tickets.groupby(client_col)[esc_flag].apply(
            lambda s: s.astype(str).str.lower().isin(["yes","true","1","y","escalated"]).mean()*100
        )

    display(client_summary.sort_values("tickets", ascending=False).head(20))

    client_summary.sort_values("tickets", ascending=False).head(15)["tickets"].plot(kind="bar", title="Top Clients by Ticket Volume")
    plt.xlabel("Client")
    plt.ylabel("Tickets")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No client column found automatically.")


## 14. Agent analysis — use carefully

Agent-level analysis can reveal workload imbalance or process differences, but **do not immediately interpret an agent's high ticket count or slower time as poor performance**. Workload, ticket mix, priority, and complexity can differ.

Useful questions:
- Are some agents handling much larger volumes?
- Are there major differences after controlling for category/priority?
- Are there process patterns worth learning from high-performing agents?


In [ ]:
agent_col = find_col(["agent", "agent_name", "assigned_agent"])

if agent_col:
    agent_summary = tickets.groupby(agent_col, dropna=False).agg(
        tickets=(tickets.columns[0], "size")
    )
    if "resolution_hours" in tickets.columns:
        agent_summary["median_resolution_h"] = tickets.groupby(agent_col)["resolution_hours"].median()
    if esc_flag:
        agent_summary["escalation_rate_pct"] = tickets.groupby(agent_col)[esc_flag].apply(
            lambda s: s.astype(str).str.lower().isin(["yes","true","1","y","escalated"]).mean()*100
        )
    display(agent_summary.sort_values("tickets", ascending=False))
else:
    print("No agent column found automatically.")


## 15. CSAT

CSAT is a guardrail against optimizing the backlog at the expense of customer experience.

Check:
- overall CSAT
- CSAT by category
- CSAT by priority
- CSAT by escalation
- CSAT versus resolution time/reopen status, where supported



In [ ]:
csat_col = find_col(["csat", "customer_satisfaction", "satisfaction_score"])

if csat_col:
    print("CSAT field:", csat_col)
    display(tickets[csat_col].describe())
    display(tickets[csat_col].value_counts(dropna=False).sort_index().to_frame("count"))

    for group in ["category", "priority"]:
        if group in tickets.columns:
            print(f"\nCSAT by {group}:")
            display(tickets.groupby(group, dropna=False)[csat_col].agg(["count", "mean"]).sort_values("mean"))
else:
    print("No CSAT column found automatically.")


## 16. Build a root-cause evidence table

Do not choose the v0 from intuition alone.

Fill this table after reviewing the outputs above.

For each hypothesis, record:
- the metric
- the observed value
- why it matters
- whether the data supports or weakens the hypothesis

Examples of hypotheses to test:
- Incoming ticket volume is the main driver.
- Resolution/closure is too slow.
- Engineering receives too many tickets.
- Engineering turnaround is the main bottleneck.
- Many escalations return to L1.
- Reopens create additional workload.
- A small number of categories drive a disproportionate share.
- A small number of clients drive a disproportionate share.


In [ ]:
# Empty template for your conclusions.
root_cause_evidence = pd.DataFrame([
    ["Incoming volume", "", "", ""],
    ["Slow resolution/closure", "", "", ""],
    ["High escalation rate", "", "", ""],
    ["Escalation returned to L1", "", "", ""],
    ["Engineering turnaround", "", "", ""],
    ["Reopens", "", "", ""],
    ["Category concentration", "", "", ""],
    ["Client concentration", "", "", ""],
], columns=["Hypothesis", "Evidence / metric", "Observed value", "Supports? Why?"])

display(root_cause_evidence)


# 17. Decision checkpoint — do not build yet

At this point, write a short evidence-based diagnosis.

Complete these sentences:

1. **The backlog is primarily driven by:** ______
2. **The strongest evidence is:** ______
3. **The biggest operational bottleneck is:** ______
4. **The user I should help first is:** ______
5. **The metric my v0 should move is:** ______
6. **I am deliberately NOT solving:** ______

Only after these are supported by the data should we design the v0.

### Product principle

> **One user → one bottleneck → one workflow → one primary metric.**

Do not build a client chatbot, agent tool, engineering dashboard, and AI system simultaneously. The case explicitly asks for one focused working v0.


# 18. Optional: export cleaned analysis data

If you create derived fields above, save a working analysis CSV for use while building the v0.

In [ ]:
OUTPUT_PATH = "rideops_tickets_eda_ready.csv"
tickets.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
